Setup

In [3]:
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from sklearn.metrics import (classification_report, f1_score,
                              recall_score, precision_score)

BASE = "C:/Users/nikes/OneDrive/Desktop/Beginner-to-Architect/aiguard/airguard-uk/airguard-uk"

results_log = []

def evaluate(model, X_test, y_test, label):
    y_pred = model.predict(X_test)
    proba  = model.predict_proba(X_test)

    macro = f1_score(y_test, y_pred, average='macro')
    dr    = recall_score(y_test, y_pred, labels=[2], average='micro')

    print(f"\n{'='*55}")
    print(f"EXPERIMENT: {label}")
    print(f"Best iteration: {model.best_iteration}")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred,
          target_names=['Low','Moderate','Dangerous'], digits=3))

    print("--- Threshold sweep ---")
    best_row = None
    for t in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
        y_t  = [2 if p[2]>=t else (1 if p[1]>=0.45 else 0)
                for p in proba]
        m    = f1_score(y_test, y_t, average='macro')
        dr_t = recall_score(y_test, y_t, labels=[2], average='micro')
        dp_t = precision_score(y_test, y_t, labels=[2],
                               average='micro', zero_division=0)
        print(f"  t={t:.2f} | Macro F1: {m:.3f} | "
              f"Dangerous recall: {dr_t*100:.1f}% | "
              f"Prec: {dp_t*100:.1f}%")
        if best_row is None or dr_t > best_row[2]:
            best_row = (t, m, dr_t, dp_t)

    results_log.append({
        'experiment':       label,
        'best_iteration':   model.best_iteration,
        'macro_f1':         round(macro, 3),
        'dangerous_recall': round(dr, 3),
        'best_threshold':   best_row[0],
        'best_macro_f1':    round(best_row[1], 3),
        'best_d_recall':    round(best_row[2], 3),
        'best_d_prec':      round(best_row[3], 3),
    })

    print(f"\n✓ Best threshold: {best_row[0]} → "
          f"Macro F1: {best_row[1]:.3f} | "
          f"Dangerous recall: {best_row[2]*100:.1f}%")
    return model

print("Setup complete.")

Setup complete.


Load data

In [4]:
train = pd.read_csv(f"{BASE}/models/train_forecast.csv")
test  = pd.read_csv(f"{BASE}/models/test_forecast.csv")

FEATURES = pd.read_csv(f"{BASE}/models/feature_list.csv")['0'].tolist()

X_train = train[FEATURES].fillna(train[FEATURES].median())
y_train_6h  = train['DAQI_6h']
y_train_12h = train['DAQI_12h']

X_test  = test[FEATURES].fillna(X_train.median())
y_test_6h   = test['DAQI_6h']
y_test_12h  = test['DAQI_12h']

print(f"Features:    {len(FEATURES)}")
print(f"Train rows:  {len(X_train):,}")
print(f"Test rows:   {len(X_test):,}")
print(f"\n6h target:\n{y_train_6h.value_counts().sort_index()}")
print(f"\n12h target:\n{y_train_12h.value_counts().sort_index()}")

Features:    150
Train rows:  174,955
Test rows:   43,745

6h target:
DAQI_6h
0    171543
1      3007
2       405
Name: count, dtype: int64

12h target:
DAQI_12h
0    171543
1      3007
2       405
Name: count, dtype: int64


Iteration 1: Baseline GPU (6h + 12h)

In [5]:
smote = SMOTE(random_state=42)
cw = {0:1, 1:5, 2:20}

# ── 6H ──────────────────────────────────────────
print("ITER 1A — 6h Baseline")
X_bal, y_bal = smote.fit_resample(X_train, y_train_6h)
sw = np.array([cw[y] for y in y_bal])

m6_i1 = XGBClassifier(
    n_estimators=1000, max_depth=6, learning_rate=0.02,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='mlogloss', early_stopping_rounds=50,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m6_i1.fit(X_bal, y_bal, sample_weight=sw,
          eval_set=[(X_test, y_test_6h)], verbose=200)
print(f"Time: {time.time()-t0:.1f}s")
evaluate(m6_i1, X_test, y_test_6h, "6h_iter1_baseline")

# ── 12H ─────────────────────────────────────────
print("\nITER 1B — 12h Baseline")
X_bal12, y_bal12 = smote.fit_resample(X_train, y_train_12h)
sw12 = np.array([cw[y] for y in y_bal12])

m12_i1 = XGBClassifier(
    n_estimators=1000, max_depth=6, learning_rate=0.02,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='mlogloss', early_stopping_rounds=50,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m12_i1.fit(X_bal12, y_bal12, sample_weight=sw12,
           eval_set=[(X_test, y_test_12h)], verbose=200)
print(f"Time: {time.time()-t0:.1f}s")
evaluate(m12_i1, X_test, y_test_12h, "12h_iter1_baseline")

ITER 1A — 6h Baseline
[0]	validation_0-mlogloss:3.11168
[200]	validation_0-mlogloss:0.97357
[400]	validation_0-mlogloss:0.66416
[600]	validation_0-mlogloss:0.49862
[800]	validation_0-mlogloss:0.39083
[999]	validation_0-mlogloss:0.31992
Time: 33.3s

EXPERIMENT: 6h_iter1_baseline
Best iteration: 999
              precision    recall  f1-score   support

         Low      0.990     0.886     0.935     42471
    Moderate      0.134     0.618     0.220      1149
   Dangerous      0.036     0.128     0.056       125

    accuracy                          0.877     43745
   macro avg      0.387     0.544     0.404     43745
weighted avg      0.965     0.877     0.914     43745

--- Threshold sweep ---
  t=0.10 | Macro F1: 0.387 | Dangerous recall: 51.2% | Prec: 1.8%
  t=0.15 | Macro F1: 0.394 | Dangerous recall: 48.0% | Prec: 2.5%
  t=0.20 | Macro F1: 0.396 | Dangerous recall: 37.6% | Prec: 2.8%
  t=0.25 | Macro F1: 0.399 | Dangerous recall: 31.2% | Prec: 3.1%
  t=0.30 | Macro F1: 0.399 | Dan

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

Iteration 2: Deeper trees + more estimators

In [6]:
# ── 6H ──────────────────────────────────────────
print("ITER 2A — 6h Deep trees")
m6_i2 = XGBClassifier(
    n_estimators=2000, max_depth=8, learning_rate=0.01,
    subsample=0.9, colsample_bytree=0.9, colsample_bylevel=0.8,
    min_child_weight=3, gamma=0.1, reg_alpha=0.1, reg_lambda=1.5,
    eval_metric='mlogloss', early_stopping_rounds=75,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m6_i2.fit(X_bal, y_bal, sample_weight=sw,
          eval_set=[(X_test, y_test_6h)], verbose=200)
print(f"Time: {time.time()-t0:.1f}s")
evaluate(m6_i2, X_test, y_test_6h, "6h_iter2_deep")

# ── 12H ─────────────────────────────────────────
print("\nITER 2B — 12h Deep trees")
m12_i2 = XGBClassifier(
    n_estimators=2000, max_depth=8, learning_rate=0.01,
    subsample=0.9, colsample_bytree=0.9, colsample_bylevel=0.8,
    min_child_weight=3, gamma=0.1, reg_alpha=0.1, reg_lambda=1.5,
    eval_metric='mlogloss', early_stopping_rounds=75,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m12_i2.fit(X_bal12, y_bal12, sample_weight=sw12,
           eval_set=[(X_test, y_test_12h)], verbose=200)
print(f"Time: {time.time()-t0:.1f}s")
evaluate(m12_i2, X_test, y_test_12h, "12h_iter2_deep")

ITER 2A — 6h Deep trees
[0]	validation_0-mlogloss:3.14866
[200]	validation_0-mlogloss:1.01336
[400]	validation_0-mlogloss:0.66237
[600]	validation_0-mlogloss:0.49707
[800]	validation_0-mlogloss:0.38861
[1000]	validation_0-mlogloss:0.32098
[1200]	validation_0-mlogloss:0.27257
[1400]	validation_0-mlogloss:0.23802
[1600]	validation_0-mlogloss:0.21237
[1800]	validation_0-mlogloss:0.19272
[1999]	validation_0-mlogloss:0.17734
Time: 108.5s

EXPERIMENT: 6h_iter2_deep
Best iteration: 1999
              precision    recall  f1-score   support

         Low      0.985     0.950     0.967     42471
    Moderate      0.213     0.494     0.298      1149
   Dangerous      0.034     0.040     0.037       125

    accuracy                          0.935     43745
   macro avg      0.411     0.495     0.434     43745
weighted avg      0.962     0.935     0.947     43745

--- Threshold sweep ---
  t=0.10 | Macro F1: 0.438 | Dangerous recall: 27.2% | Prec: 4.1%
  t=0.15 | Macro F1: 0.436 | Dangerous recal

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,0.8
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",75
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=

Iteration 3: SMOTETomek + stronger weights

In [7]:
st = SMOTETomek(random_state=42)
cw3 = {0:1, 1:8, 2:30}

# ── 6H ──────────────────────────────────────────
print("ITER 3A — 6h SMOTETomek")
X_bal3, y_bal3 = st.fit_resample(X_train, y_train_6h)
sw3 = np.array([cw3[y] for y in y_bal3])

m6_i3 = XGBClassifier(
    n_estimators=1500, max_depth=7, learning_rate=0.015,
    subsample=0.9, colsample_bytree=0.9, colsample_bylevel=0.85,
    min_child_weight=2, gamma=0.05, reg_alpha=0.2, reg_lambda=1.5,
    eval_metric='mlogloss', early_stopping_rounds=60,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m6_i3.fit(X_bal3, y_bal3, sample_weight=sw3,
          eval_set=[(X_test, y_test_6h)], verbose=200)
print(f"Time: {time.time()-t0:.1f}s")
evaluate(m6_i3, X_test, y_test_6h, "6h_iter3_smotetomek")

# ── 12H ─────────────────────────────────────────
print("\nITER 3B — 12h SMOTETomek")
X_bal3_12, y_bal3_12 = st.fit_resample(X_train, y_train_12h)
sw3_12 = np.array([cw3[y] for y in y_bal3_12])

m12_i3 = XGBClassifier(
    n_estimators=1500, max_depth=7, learning_rate=0.015,
    subsample=0.9, colsample_bytree=0.9, colsample_bylevel=0.85,
    min_child_weight=2, gamma=0.05, reg_alpha=0.2, reg_lambda=1.5,
    eval_metric='mlogloss', early_stopping_rounds=60,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m12_i3.fit(X_bal3_12, y_bal3_12, sample_weight=sw3_12,
           eval_set=[(X_test, y_test_12h)], verbose=200)
print(f"Time: {time.time()-t0:.1f}s")
evaluate(m12_i3, X_test, y_test_12h, "12h_iter3_smotetomek")

ITER 3A — 6h SMOTETomek
[0]	validation_0-mlogloss:3.49857
[200]	validation_0-mlogloss:1.06282
[400]	validation_0-mlogloss:0.72514
[600]	validation_0-mlogloss:0.52938
[800]	validation_0-mlogloss:0.41712
[1000]	validation_0-mlogloss:0.34070
[1200]	validation_0-mlogloss:0.28781
[1400]	validation_0-mlogloss:0.24704
[1499]	validation_0-mlogloss:0.23260
Time: 63.5s

EXPERIMENT: 6h_iter3_smotetomek
Best iteration: 1499
              precision    recall  f1-score   support

         Low      0.988     0.925     0.955     42471
    Moderate      0.174     0.568     0.267      1149
   Dangerous      0.050     0.080     0.062       125

    accuracy                          0.914     43745
   macro avg      0.404     0.525     0.428     43745
weighted avg      0.964     0.914     0.935     43745

--- Threshold sweep ---
  t=0.10 | Macro F1: 0.416 | Dangerous recall: 31.2% | Prec: 2.9%
  t=0.15 | Macro F1: 0.422 | Dangerous recall: 27.2% | Prec: 3.8%
  t=0.20 | Macro F1: 0.425 | Dangerous recall: 

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,0.85
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",60
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

Iteration 4: Best config + colsample_bylevel tuning

In [8]:
# ── 6H ──────────────────────────────────────────
print("ITER 4A — 6h High regularisation")
m6_i4 = XGBClassifier(
    n_estimators=2000, max_depth=7, learning_rate=0.01,
    subsample=0.9, colsample_bytree=0.85, colsample_bylevel=0.75,
    colsample_bynode=0.8,
    min_child_weight=4, gamma=0.15, reg_alpha=0.3, reg_lambda=2.0,
    eval_metric='mlogloss', early_stopping_rounds=75,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m6_i4.fit(X_bal, y_bal, sample_weight=sw,
          eval_set=[(X_test, y_test_6h)], verbose=200)
print(f"Time: {time.time()-t0:.1f}s")
evaluate(m6_i4, X_test, y_test_6h, "6h_iter4_high_reg")

# ── 12H ─────────────────────────────────────────
print("\nITER 4B — 12h High regularisation")
m12_i4 = XGBClassifier(
    n_estimators=2000, max_depth=7, learning_rate=0.01,
    subsample=0.9, colsample_bytree=0.85, colsample_bylevel=0.75,
    colsample_bynode=0.8,
    min_child_weight=4, gamma=0.15, reg_alpha=0.3, reg_lambda=2.0,
    eval_metric='mlogloss', early_stopping_rounds=75,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m12_i4.fit(X_bal12, y_bal12, sample_weight=sw12,
           eval_set=[(X_test, y_test_12h)], verbose=200)
print(f"Time: {time.time()-t0:.1f}s")
evaluate(m12_i4, X_test, y_test_12h, "12h_iter4_high_reg")

ITER 4A — 6h High regularisation
[0]	validation_0-mlogloss:3.15525
[200]	validation_0-mlogloss:1.16533
[400]	validation_0-mlogloss:0.80254
[600]	validation_0-mlogloss:0.63566
[800]	validation_0-mlogloss:0.51088
[1000]	validation_0-mlogloss:0.43150
[1200]	validation_0-mlogloss:0.37048
[1400]	validation_0-mlogloss:0.32501
[1600]	validation_0-mlogloss:0.28874
[1800]	validation_0-mlogloss:0.26046
[1999]	validation_0-mlogloss:0.23696
Time: 101.9s

EXPERIMENT: 6h_iter4_high_reg
Best iteration: 1999
              precision    recall  f1-score   support

         Low      0.988     0.920     0.953     42471
    Moderate      0.164     0.560     0.253      1149
   Dangerous      0.037     0.072     0.049       125

    accuracy                          0.909     43745
   macro avg      0.396     0.517     0.419     43745
weighted avg      0.963     0.909     0.932     43745

--- Threshold sweep ---
  t=0.10 | Macro F1: 0.410 | Dangerous recall: 36.8% | Prec: 2.6%
  t=0.15 | Macro F1: 0.413 | Da

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,0.75
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,0.8
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.85
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",75
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

Iter 5: Targeted fix

In [10]:
# ── New evaluate with extended low-end threshold sweep ──────────────
def evaluate_v2(model, X_test, y_test, label):
    y_pred = model.predict(X_test)
    proba  = model.predict_proba(X_test)

    macro = f1_score(y_test, y_pred, average='macro')
    dr    = recall_score(y_test, y_pred, labels=[2], average='micro')

    print(f"\n{'='*55}")
    print(f"EXPERIMENT: {label}")
    print(f"Best iteration: {model.best_iteration}")
    print(f"{'='*55}")
    print(classification_report(y_test, y_pred,
          target_names=['Low','Moderate','Dangerous'], digits=3))

    print("--- Threshold sweep ---")
    best_row = None
    for t in [0.04, 0.06, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25]:
        y_t  = [2 if p[2]>=t else (1 if p[1]>=0.45 else 0)
                for p in proba]
        m    = f1_score(y_test, y_t, average='macro')
        dr_t = recall_score(y_test, y_t, labels=[2], average='micro')
        dp_t = precision_score(y_test, y_t, labels=[2],
                               average='micro', zero_division=0)
        print(f"  t={t:.2f} | Macro F1: {m:.3f} | "
              f"Dangerous recall: {dr_t*100:.1f}% | "
              f"Prec: {dp_t*100:.1f}%")
        if best_row is None or dr_t > best_row[2]:
            best_row = (t, m, dr_t, dp_t)

    results_log.append({
        'experiment':       label,
        'best_iteration':   model.best_iteration,
        'macro_f1':         round(macro, 3),
        'dangerous_recall': round(dr, 3),
        'best_threshold':   best_row[0],
        'best_macro_f1':    round(best_row[1], 3),
        'best_d_recall':    round(best_row[2], 3),
        'best_d_prec':      round(best_row[3], 3),
    })

    print(f"\n✓ Best threshold: {best_row[0]} → "
          f"Macro F1: {best_row[1]:.3f} | "
          f"Dangerous recall: {best_row[2]*100:.1f}%")
    return model

# ── New weights — much harder push on Dangerous ─────────────────────
cw5 = {0:1, 1:10, 2:50}
sw5    = np.array([cw5[y] for y in y_bal])    # reuse Iter 1 SMOTE data
sw5_12 = np.array([cw5[y] for y in y_bal12])

# ── 6H ───────────────────────────────────────────────────────────────
print("ITER 5A — 6h Targeted (aucpr + heavier weights)")
m6_i5 = XGBClassifier(
    n_estimators=3000, max_depth=6, learning_rate=0.02,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='aucpr', early_stopping_rounds=100,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m6_i5.fit(X_bal, y_bal, sample_weight=sw5,
          eval_set=[(X_test, y_test_6h)], verbose=300)
print(f"Time: {time.time()-t0:.1f}s")
evaluate_v2(m6_i5, X_test, y_test_6h, "6h_iter5_targeted")

# ── 12H ──────────────────────────────────────────────────────────────
print("\nITER 5B — 12h Targeted (aucpr + heavier weights)")
m12_i5 = XGBClassifier(
    n_estimators=3000, max_depth=6, learning_rate=0.02,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='aucpr', early_stopping_rounds=100,
    device='cuda', tree_method='hist', random_state=42, n_jobs=-1
)
t0 = time.time()
m12_i5.fit(X_bal12, y_bal12, sample_weight=sw5_12,
           eval_set=[(X_test, y_test_12h)], verbose=300)
print(f"Time: {time.time()-t0:.1f}s")
evaluate_v2(m12_i5, X_test, y_test_12h, "12h_iter5_targeted")

ITER 5A — 6h Targeted (aucpr + heavier weights)
[0]	validation_0-aucpr:0.35186
[300]	validation_0-aucpr:0.38582
[600]	validation_0-aucpr:0.39685
[900]	validation_0-aucpr:0.40655
[1200]	validation_0-aucpr:0.41312
[1500]	validation_0-aucpr:0.41860
[1800]	validation_0-aucpr:0.42269
[2100]	validation_0-aucpr:0.42596
[2400]	validation_0-aucpr:0.42861
[2700]	validation_0-aucpr:0.43077
[2999]	validation_0-aucpr:0.43259
Time: 97.7s

EXPERIMENT: 6h_iter5_targeted
Best iteration: 2997
              precision    recall  f1-score   support

         Low      0.984     0.966     0.975     42471
    Moderate      0.270     0.459     0.340      1149
   Dangerous      0.042     0.024     0.030       125

    accuracy                          0.950     43745
   macro avg      0.432     0.483     0.449     43745
weighted avg      0.962     0.950     0.956     43745

--- Threshold sweep ---
  t=0.04 | Macro F1: 0.449 | Dangerous recall: 19.2% | Prec: 3.4%
  t=0.06 | Macro F1: 0.450 | Dangerous recall: 15

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",100
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

Results table + save best models

In [11]:
import joblib, json, os

# Add iter 5 models to the map (others already trained above)
model_map['6h_iter5_targeted']  = m6_i5
model_map['12h_iter5_targeted'] = m12_i5

results_df = pd.DataFrame(results_log)
print("\n" + "="*80)
print("FULL RESULTS TABLE — ALL ITERATIONS")
print("="*80)
print(results_df[[
    'experiment','best_iteration','macro_f1',
    'dangerous_recall','best_threshold',
    'best_macro_f1','best_d_recall','best_d_prec'
]].to_string(index=False))

# Best per horizon — ranked by Dangerous recall
best_6h  = results_df[results_df['experiment'].str.startswith('6h')] \
    .sort_values('best_d_recall', ascending=False).iloc[0]
best_12h = results_df[results_df['experiment'].str.startswith('12h')] \
    .sort_values('best_d_recall', ascending=False).iloc[0]

print(f"\n{'='*40}")
print(f"BEST 6h:  {best_6h['experiment']}")
print(f"  Macro F1: {best_6h['best_macro_f1']} | "
      f"Dangerous recall: {best_6h['best_d_recall']*100:.1f}% | "
      f"Threshold: {best_6h['best_threshold']}")
print(f"\nBEST 12h: {best_12h['experiment']}")
print(f"  Macro F1: {best_12h['best_macro_f1']} | "
      f"Dangerous recall: {best_12h['best_d_recall']*100:.1f}% | "
      f"Threshold: {best_12h['best_threshold']}")

best_6h_model  = model_map[best_6h['experiment']]
best_12h_model = model_map[best_12h['experiment']]

joblib.dump(best_6h_model,  f"{BASE}/models/xgboost_6h_best.pkl")
joblib.dump(best_12h_model, f"{BASE}/models/xgboost_12h_best.pkl")

thresholds = {
    '6h':  float(best_6h['best_threshold']),
    '12h': float(best_12h['best_threshold']),
}
with open(f"{BASE}/models/thresholds.json", 'w') as f:
    json.dump(thresholds, f)

results_df.to_csv(f"{BASE}/models/tuning_results.csv", index=False)

print(f"\n✓ models/xgboost_6h_best.pkl  → {best_6h['experiment']}")
print(f"✓ models/xgboost_12h_best.pkl → {best_12h['experiment']}")
print(f"✓ models/thresholds.json      → {thresholds}")
print(f"✓ models/tuning_results.csv   — {len(results_df)} experiments")


FULL RESULTS TABLE — ALL ITERATIONS
          experiment  best_iteration  macro_f1  dangerous_recall  best_threshold  best_macro_f1  best_d_recall  best_d_prec
   6h_iter1_baseline             999     0.404             0.128            0.10          0.387          0.512        0.018
  12h_iter1_baseline             999     0.395             0.096            0.10          0.380          0.432        0.013
       6h_iter2_deep            1999     0.434             0.040            0.10          0.438          0.272        0.041
      12h_iter2_deep            1999     0.420             0.040            0.10          0.419          0.152        0.025
 6h_iter3_smotetomek            1499     0.428             0.080            0.10          0.416          0.312        0.029
12h_iter3_smotetomek            1499     0.414             0.064            0.10          0.406          0.232        0.020
   6h_iter4_high_reg            1999     0.419             0.072            0.10          0.410